# Step 2 — Zero-shot annotation with a Vision-Language Model

**The pitch.** Labeling images by hand is slow and expensive. A Vision-Language Model like **Qwen2.5-VL** already "understands" what a `helmet`, `vest`, `gloves`, or `person` looks like. We ask it, in natural language, to return a structured JSON list of bounding boxes — and we write those boxes straight into FiftyOne as labels.

At the end of this notebook, the 40 training images from `datasets/ppe/images/train/` have four-class PPE annotations that are good enough to fine-tune a specialist YOLO detector on in the next step.

**Taxonomy:** `person`, `helmet`, `vest`, `gloves`.

**Prereq:** notebook 01 has run and the FiftyOne dataset `ppe` (40 images) exists locally.

In [1]:
%%capture
!pip install matplotlib pillow requests tqdm fiftyone gradio

In [8]:
import base64
import json
import mimetypes
import os
import re
from pathlib import Path

import fiftyone as fo
import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

## Configuration

`ENDPOINT` points at a Qwen2.5-VL vLLM service exposed as an OpenAI-compatible route. Auth uses a bearer token from `VLLM_API_TOKEN` or `OPENAI_API_KEY`.

In [9]:
ENDPOINT = os.environ.get(
    "VLM_ENDPOINT",
    "https://qwen25-vl-7b-instruct-ppe-detection-cv.apps.ocp.9xgvv.sandbox3434.opentlc.com",
)

CLASSES = ["person", "helmet", "vest", "gloves"]

LABEL_COLORS = {
    "person": (255, 0, 0),
    "helmet": (255, 165, 0),
    "vest":   (255, 255, 0),
    "gloves": (0, 255, 255),
}

# Tag marking samples the VLM has annotated — picked up by notebook 03.
VLM_TAG = "vlm_annotated"

NOTEBOOK_DIR = Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

## VLM client helpers

In [10]:
def auth_headers() -> dict:
    token = os.environ.get("VLLM_API_TOKEN") or os.environ.get("OPENAI_API_KEY", "external-token")
    return {"Authorization": f"Bearer {token}"} if token else {}


def encode_image(path: Path) -> str:
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    b64 = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{b64}"


def get_model(base_url: str) -> str:
    r = requests.get(f"{base_url}/v1/models", timeout=30, headers=auth_headers())
    r.raise_for_status()
    return r.json()["data"][0]["id"]


PROMPT = (
    "Detect every person and every PPE item in the image. "
    f"Only use these four labels: {', '.join(CLASSES)}. "
    "Return ONLY valid JSON, no prose, no markdown fences. Schema:\n"
    "{\n"
    '  "image_size": {"width": <int>, "height": <int>},\n'
    '  "detections": [\n'
    '    {"label": "<one of: person|helmet|vest|gloves>", "bbox_2d": [x1, y1, x2, y2], "confidence": <float 0-1>}\n'
    "  ]\n"
    "}\n"
    "Use absolute pixel coordinates: (x1,y1)=top-left, (x2,y2)=bottom-right."
)


def infer(image_path: Path, base_url: str, model: str, prompt: str = PROMPT) -> str:
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": encode_image(image_path)}},
                ],
            }
        ],
        "max_tokens": 1024,
        "temperature": 0.2,
    }
    r = requests.post(
        f"{base_url}/v1/chat/completions",
        json=payload,
        timeout=180,
        headers={"Content-Type": "application/json", **auth_headers()},
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def parse_detections(raw: str) -> dict:
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in model output: {raw[:200]}")
    return json.loads(match.group(0))

## Interactive playground — try your own labels

Before we run the VLM programmatically on the 40 training images, let's prove the endpoint is alive and responsive by poking at it interactively. This Gradio app lets anyone type arbitrary labels (e.g., `worker, safety helmet, reflective vest` or `ladder, cone, truck`) and see what the VLM finds on one of three pre-loaded samples.

**Outputs:**
1. The input image with the detected bounding boxes drawn on top.
2. A text panel showing the VLM's one-line scene description plus each detection's label, confidence, and pixel-space bbox.

In [11]:
import hashlib

import gradio as gr

# Try to fetch the serving model ID up-front. If the endpoint is down / SSL-broken /
# the token is wrong, keep MODEL_ID = None and let gradio_detect report the error
# on Submit — this way the app still launches so the attendee can see the UI.
MODEL_ID: str | None = None
MODEL_ID_ERROR: str | None = None
try:
    MODEL_ID = get_model(ENDPOINT)
    print("Serving model:", MODEL_ID)
except Exception as e:
    MODEL_ID_ERROR = f"{type(e).__name__}: {e}"
    print(f"\u26a0 VLM endpoint unreachable: {MODEL_ID_ERROR}")
    print("  Gradio will launch anyway and surface this error on Submit.")


def make_prompt(labels: list[str]) -> str:
    cls_str = ", ".join(labels) if labels else "any salient objects"
    return (
        f"Detect every instance of the following in the image: {cls_str}. "
        f"Only use these exact labels (one per detection): {cls_str}. "
        "Return ONLY valid JSON, no prose, no markdown fences. Schema:\n"
        "{\n"
        '  "image_size": {"width": <int>, "height": <int>},\n'
        '  "description": "<one short sentence summarizing the scene>",\n'
        '  "detections": [\n'
        f'    {{"label": "<one of: {cls_str}>", "bbox_2d": [x1, y1, x2, y2], "confidence": <float 0-1>}}\n'
        "  ]\n"
        "}\n"
        "Use absolute pixel coordinates: (x1,y1)=top-left, (x2,y2)=bottom-right."
    )


def color_for(label: str) -> tuple:
    if label in LABEL_COLORS:
        return LABEL_COLORS[label]
    h = hashlib.md5(label.encode()).digest()
    return (55 + h[0] % 200, 55 + h[1] % 200, 55 + h[2] % 200)


def draw_any(image_path: Path, data: dict) -> Image.Image:
    """Like draw_detections, but accepts any label (no CLASSES filter)."""
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 14)
    except OSError:
        font = ImageFont.load_default()
    for det in data.get("detections", []):
        label = det.get("label", "?")
        x1, y1, x2, y2 = det["bbox_2d"]
        color = color_for(label)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} {det.get('confidence', 0):.2f}"
        tb = draw.textbbox((0, 0), text, font=font)
        th = tb[3] - tb[1]
        draw.rectangle([x1, max(0, y1 - th - 4), x1 + (tb[2] - tb[0]) + 6, y1], fill=color)
        draw.text((x1 + 3, max(0, y1 - th - 2)), text, fill=(0, 0, 0), font=font)
    return img


def format_description(parsed: dict, labels: list[str]) -> str:
    scene = (parsed.get("description") or "").strip()
    dets = parsed.get("detections", [])
    lines = []
    if scene:
        lines.append(f"Scene: {scene}")
        lines.append("")
    label_tag = ", ".join(labels) if labels else "(unconstrained)"
    lines.append(f"Found {len(dets)} detection(s) for labels: {label_tag}")
    for d in dets:
        lab = d.get("label", "?")
        conf = float(d.get("confidence", 0.0))
        x1, y1, x2, y2 = d.get("bbox_2d", [0, 0, 0, 0])
        lines.append(f"  \u2022 {lab:<14} conf={conf:.0%}  bbox=[{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]")
    return "\n".join(lines)


def _ensure_model_id() -> str:
    """Lazily (re)fetch the model id so a transient outage at notebook start
    doesn't permanently kill the playground."""
    global MODEL_ID, MODEL_ID_ERROR
    if MODEL_ID is not None:
        return MODEL_ID
    MODEL_ID = get_model(ENDPOINT)
    MODEL_ID_ERROR = None
    return MODEL_ID


def gradio_detect(image_path: str, labels_text: str):
    if not image_path:
        return None, "Pick an image first."
    labels = [lab.strip() for lab in labels_text.split(",") if lab.strip()]
    try:
        mid = _ensure_model_id()
    except Exception as e:
        return None, (
            f"\u26a0 VLM endpoint is not reachable.\n"
            f"  Endpoint: {ENDPOINT}\n"
            f"  Error:    {type(e).__name__}: {e}\n"
            "Check that the Qwen2.5-VL service is running and VLLM_API_TOKEN is set if required."
        )
    try:
        raw = infer(Path(image_path), ENDPOINT, mid, prompt=make_prompt(labels))
        parsed = parse_detections(raw)
    except Exception as e:
        return None, f"Inference failed: {type(e).__name__}: {e}"
    return draw_any(Path(image_path), parsed), format_description(parsed, labels)


SAMPLES_DIR = NOTEBOOK_DIR / "samples"
EXAMPLES = [
    [str(SAMPLES_DIR / "ppe-01.jpg"), "person, helmet, vest, gloves"],
    [str(SAMPLES_DIR / "ppe-02.jpg"), "person, hard hat, reflective vest"],
    [str(SAMPLES_DIR / "ppe-03.jpg"), "worker, safety helmet, gloves, boots"],
]

demo = gr.Interface(
    fn=gradio_detect,
    inputs=[
        gr.Image(type="filepath", label="Input image"),
        gr.Textbox(
            value="person, helmet, vest, gloves",
            label="Labels (comma-separated)",
            info="Type any object labels you want the VLM to detect.",
        ),
    ],
    outputs=[
        gr.Image(type="pil", label="Localization (bboxes on top)"),
        gr.Textbox(label="Description", lines=10, max_lines=20),
    ],
    examples=EXAMPLES,
    cache_examples=False,
    title="Qwen2.5-VL zero-shot detector",
    description=(
        "Pick a sample image (or upload your own), type any labels, and see what the VLM finds. "
        + ("\u26a0 Endpoint currently unreachable \u2014 Submit will show the error." if MODEL_ID is None else "")
    ),
)

# Shut down any Gradio app left running from a previous cell execution so re-running
# this cell doesn't trip over "[Errno 48] address already in use". Dropping server_port
# lets Gradio walk forward from 7860 to find a free port if something else is bound.
gr.close_all()
demo.launch(server_name="0.0.0.0", inline=True, share=False)

Serving model: qwen25-vl-7b-instruct
Closing server running on port: 7860
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


## Smoke test — one image

Confirm the endpoint is reachable and returning parseable JSON before we batch.

In [12]:
model_id = get_model(ENDPOINT)
print("Serving model:", model_id)

sample_path = NOTEBOOK_DIR / "samples" / "ppe-01.jpg"
raw = infer(sample_path, ENDPOINT, model_id)
print(raw[:500])
parsed = parse_detections(raw)
print(f"\nParsed {len(parsed.get('detections', []))} detections.")

Serving model: qwen25-vl-7b-instruct
```json
{
  "image_size": {
    "width": 640,
    "height": 480
  },
  "detections": [
    {
      "label": "person",
      "bbox_2d": [357, 139, 490, 452],
      "confidence": 0.95
    },
    {
      "label": "person",
      "bbox_2d": [183, 116, 291, 416],
      "confidence": 0.95
    },
    {
      "label": "helmet",
      "bbox_2d": [432, 139, 473, 168],
      "confidence": 0.9
    },
    {
      "label": "helmet",
      "bbox_2d": [190, 116, 232, 143],
      "confidence": 0.9
    },
    {
 

Parsed 8 detections.


## Preview: draw the VLM output on the smoke-test image

In [13]:
def draw_detections(image_path: Path, data: dict) -> Image.Image:
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 14)
    except OSError:
        font = ImageFont.load_default()
    for det in data.get("detections", []):
        label = det.get("label", "?")
        if label not in CLASSES:
            continue
        x1, y1, x2, y2 = det["bbox_2d"]
        color = LABEL_COLORS.get(label, (0, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} {det.get('confidence', 0):.2f}"
        tb = draw.textbbox((0, 0), text, font=font)
        th = tb[3] - tb[1]
        draw.rectangle([x1, max(0, y1 - th - 4), x1 + (tb[2] - tb[0]) + 6, y1], fill=color)
        draw.text((x1 + 3, max(0, y1 - th - 2)), text, fill=(0, 0, 0), font=font)
    return img


plt.figure(figsize=(10, 8))
plt.imshow(draw_detections(sample_path, parsed))
plt.axis("off")
plt.show()

<Figure size 1000x800 with 1 Axes>

## Batch annotate all 40 training images

Call the VLM on every sample, attach the result as a `vlm_predictions` field of type `fo.Detections`. FiftyOne stores bounding boxes in normalized `[x, y, w, h]` with origin at the top-left — we convert from the VLM's absolute-pixel `[x1, y1, x2, y2]`.

In [14]:
dataset = fo.load_dataset("ppe")

# Annotate only the train split. The 10 test images stay held-out for notebook 06.
train_view = dataset.match_tags("train")
to_annotate = [s.id for s in train_view.select_fields("id")]
print(f"Annotating {len(to_annotate)} train images ({len(dataset) - len(to_annotate)} test images held out).")

Annotating 40 train images (10 test images held out).


In [ ]:
def to_fiftyone_detections(parsed: dict, img_w: int, img_h: int) -> fo.Detections:
    dets = []
    for d in parsed.get("detections", []):
        label = d.get("label")
        if label not in CLASSES:
            continue
        x1, y1, x2, y2 = d["bbox_2d"]
        x1, x2 = sorted([max(0, min(img_w, x1)), max(0, min(img_w, x2))])
        y1, y2 = sorted([max(0, min(img_h, y1)), max(0, min(img_h, y2))])
        w, h = (x2 - x1) / img_w, (y2 - y1) / img_h
        if w <= 0 or h <= 0:
            continue
        dets.append(fo.Detection(
            label=label,
            bounding_box=[x1 / img_w, y1 / img_h, w, h],
            confidence=float(d.get("confidence", 0.0)),
        ))
    return fo.Detections(detections=dets)


failures = []
for sid in tqdm(to_annotate):
    sample = dataset[sid]
    try:
        sample.compute_metadata()
        img_w, img_h = sample.metadata.width, sample.metadata.height
        raw = infer(Path(sample.filepath), ENDPOINT, model_id)
        parsed = parse_detections(raw)
        sample["vlm_predictions"] = to_fiftyone_detections(parsed, img_w, img_h)
        sample.tags = list(set(sample.tags) | {VLM_TAG})
        sample.save()
    except Exception as e:
        failures.append((sid, str(e)))

print(f"\nAnnotated {len(to_annotate) - len(failures)}/{len(to_annotate)} samples. {len(failures)} failures.")
if failures[:3]:
    print("First failures:", failures[:3])

  8%|▊         | 3/40 [00:22<05:02,  8.17s/it]

## Sanity check — show 6 random VLM-annotated samples

In [ ]:
annotated = dataset.match_tags(VLM_TAG)
shuffled = annotated.shuffle(seed=51).limit(6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, sample in zip(axes.flat, shuffled):
    sample.compute_metadata()
    w, h = sample.metadata.width, sample.metadata.height
    data = {"detections": []}
    for det in sample.vlm_predictions.detections:
        x, y, bw, bh = det.bounding_box
        data["detections"].append({
            "label": det.label,
            "bbox_2d": [x * w, y * h, (x + bw) * w, (y + bh) * h],
            "confidence": det.confidence,
        })
    ax.imshow(draw_detections(Path(sample.filepath), data))
    ax.set_title(Path(sample.filepath).name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

---

**What just happened.** Qwen2.5-VL — a general-purpose VLM that has never been fine-tuned on construction PPE — produced plausible bounding boxes on all 40 images with no training data, no labeled examples, and a 10-line prompt.

**Next:** [03-curate-and-export.ipynb](03-curate-and-export.ipynb) — open the FiftyOne app to visually QA the VLM labels, then export the annotated set to YOLO format for training.